# Module Development

This notebook is the local playground for the `string2BMT` package.

Goals:
- test the lightweight public API
- compare deterministic fallback vs agent-backed mapping


In [ ]:
import sys
print(sys.executable)


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from string2BMT import string_to_bmt

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


## Deterministic Preview


In [ ]:
sample_inputs = [
    '더 뉴 스파크 프리미어',
    '스파크 LS',
    '리얼 뉴 콜로라도 Z71-X',
    '볼트 euv 프리미어',
]

rows = []
for raw_name in sample_inputs:
    brand, model_name, trim_name = string_to_bmt(raw_name, use_agent=False)
    rows.append(
        {
            'raw_name': raw_name,
            'brand': brand,
            'model_name': model_name,
            'trim_name': trim_name,
        }
    )

display(pd.DataFrame(rows))


## Optional Agent Run

Copy `.env.example` to `.env` in this directory and fill `OPENROUTER_API_KEY` before enabling this section.


In [ ]:
RUN_AGENT = False
AGENT_INPUT = '더 뉴 스파크 프리미어'

if not RUN_AGENT:
    print('Set RUN_AGENT = True after preparing .env to execute the agent-backed mapper.')
else:
    string_to_bmt(AGENT_INPUT, use_agent=True)


## Agent Assertion Cases

Append more rows to `AGENT_TEST_CASES` below when you add more validation inputs.


In [ ]:
AGENT_TEST_CASES = [
    {
        'raw_name': '더 뉴 스파크 프리미어',
        'expected': ('chevrolet', '더 뉴 스파크', '프리미어'),
    },
    {
        'raw_name': '스파크 LS',
        'expected': ('chevrolet', '스파크', 'LS'),
    },
    {
        'raw_name': '리얼 뉴 콜로라도 Z71-X',
        'expected': ('chevrolet', '리얼 뉴 콜로라도', 'Z71-X'),
    },
    {
        'raw_name': '볼트 euv 프리미어',
        'expected': ('chevrolet', '볼트 EUV', '프리미어'),
    },
    {
        'raw_name': '현대 그랜저 HG HG240 모던',
        'expected': ('hyundai', '그랜저HG', '모던'),
    },
    {
        'raw_name': '현대 소나타 1.6T 인스퍼레이션',
        'expected': ('hyundai', '쏘나타 뉴 라이즈', '인스퍼레이션'),
    },
    {
        'raw_name': '기아 더 뉴 쏘렌토 MQ4 2.2 디젤 2WD 노블레스',
        'expected': ('kia', '더 뉴 쏘렌토', '노블레스'),
    },
    {
        'raw_name': '기아 올 뉴 니로 1.6 HEV 트렌디',
        'expected': ('kia', '디 올 뉴 니로', '트렌디'),
    },
    {
        'raw_name': '기아 올 뉴 K7 3.0 GDI 리미티드',
        'expected': ('kia', '올 뉴 K7', '리미티드'),
    },
]

pd.DataFrame(AGENT_TEST_CASES)


In [ ]:
if not RUN_AGENT:
    print('Set RUN_AGENT = True to run the assertion suite with the agent path.')
else:
    result_rows = []
    for case in AGENT_TEST_CASES:
        actual = string_to_bmt(case['raw_name'], use_agent=True)
        assert actual == case['expected'], (
            f"Mismatch for {case['raw_name']}: expected {case['expected']}, got {actual}"
        )
        result_rows.append(
            {
                'raw_name': case['raw_name'],
                'expected': case['expected'],
                'actual': actual,
                'status': 'passed',
            }
        )

    display(pd.DataFrame(result_rows))
